# SKANN-SSL V2.1.0 Training Notebook

**Version:** 2.1.0 (Underwater-Appropriate Kernels)  
**Date:** 2026-01-07

## Critical Fix: Kernel Sizes for Underwater Acoustics

**V2.0.x Problem:** SK kernels (3, 5, 7, 11, 15) only captured HIGH frequencies (1-5 kHz).
But vessel classification depends on LOW frequencies:

| Phenomenon | Frequency | Required Kernel | V2.0.x | V2.1.0 |
|------------|-----------|-----------------|--------|--------|
| shaft_rate | 1-30 Hz | k=500-16000 | ❌ | ✅ (1023) |
| blade_pass | 3-150 Hz | k=100-5300 | ❌ | ✅ (127-1023) |
| generator | 25, 50 Hz | k=320-640 | ❌ | ✅ (255-1023) |
| resonance | 50-500 Hz | k=31-63 | ⚠️ | ✅ (31-127) |
| cavitation | 400-6300 Hz | k=15-31 | ✅ | ✅ (31) |

## V2.1.0 Changes

1. **SK Kernel Sizes:** (3,5,7,11,15) → **(31, 63, 127, 255, 511, 1023)**
2. **Projector:** 512→1024→128 → **512→4096→8192→128** (restore V1's large projector)
3. **Keep:** SyncBatchNorm, all DDP fixes

## Kernel Coverage @ 16kHz

| Kernel | Duration | Frequency Range |
|--------|----------|----------------|
| 31 | 2 ms | Cavitation (500+ Hz) |
| 63 | 4 ms | Resonance (250+ Hz) |
| 127 | 8 ms | Blade pass (125+ Hz) |
| 255 | 16 ms | Generator 50Hz (62+ Hz) |
| 511 | 32 ms | Generator 25Hz (31+ Hz) |
| 1023 | 64 ms | Shaft rate (15+ Hz) |

### Requirements / compatibility (auto-restart only if needed)

- Ensures compatible versions of `scikit-learn`, `umap-learn`, and `joblib` **before** any heavy imports or training. If an upgrade occurs, the kernel restarts once so the new versions take effect.


In [ ]:
# Cell 0: Requirements / compatibility (auto-restart only if needed)
# This is CPU/disk-bound and does not consume GPU VRAM.

import sys, subprocess, importlib.util

def _pkg_version(pkg: str):
    try:
        mod = __import__(pkg)
        return getattr(mod, '__version__', None)
    except Exception:
        return None

REQUIRED = {
    'scikit-learn': '1.2',
    'umap-learn': '0.5.5',
    'joblib': '1.2',
}

def _ver_tuple(v: str):
    # Compare only major.minor to avoid strict patch coupling
    parts = (v or '').split('.')
    if len(parts) < 2:
        return (0, 0)
    try:
        return (int(parts[0]), int(parts[1]))
    except Exception:
        return (0, 0)

needs_install = False
for pkg, min_ver in REQUIRED.items():
    name = pkg.replace('-', '_')
    try:
        spec = importlib.util.find_spec(name)
        if spec is None:
            needs_install = True
            break
        ver = _pkg_version(name)
        if ver is None or _ver_tuple(ver) < _ver_tuple(min_ver):
            needs_install = True
            break
    except Exception:
        needs_install = True
        break

if needs_install:
    print('📦 Installing/upgrading required packages…')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'scikit-learn>=1.2',
        'umap-learn>=0.5.5',
        'joblib>=1.2',
    ])
    print('🔁 Restarting kernel once to activate new versions…')
    import os
    os._exit(0)
else:
    print('✅ Required packages already satisfied')

# Provenance (non-fatal)
try:
    import sklearn, umap, joblib
    print('sklearn:', sklearn.__version__)
    print('umap-learn:', umap.__version__)
    print('joblib:', joblib.__version__)
except Exception as e:
    print('⚠️ Version check skipped:', e)


### Environment Setup

- Clones the SKANN-SSL repository, sets up paths, and prepares the Kaggle working directory for outputs.


In [ ]:
# Cell 1: Environment Setup
import os
import gc
import sys
import time
import datetime
import traceback

def setup_environment():
    print("🧹 Environment Setup...")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29500'
    os.makedirs("/kaggle/working/run_logs", exist_ok=True)
    gc.collect()
    print("✅ Environment configured.")

def safe_cleanup():
    import torch
    print("🧹 Gentle cleanup...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    print("✅ Cleanup done.")

def log(msg, rank=None):
    ts = time.strftime('%H:%M:%S')
    if rank is not None:
        print(f"[{ts}][R{rank}] {msg}", flush=True)
    else:
        print(f"[{ts}] {msg}", flush=True)

setup_environment()
safe_cleanup()
log("✅ All utilities loaded and environment ready")

### Automated Data Ingestion

- Copies/loads the dataset inputs into the expected repo structure and produces the pairing manifest used for positive/negative pairing during SSL.


In [ ]:
# Cell 2: Automated Data Ingestion
import os
import gc

log("Starting data ingestion...")

!wget -q -O /kaggle/working/pairing_manifest.csv https://raw.githubusercontent.com/suniltyagi/SKANN-SSL/main/data/prototype_dataset/pairing_manifest.csv
log("Manifest downloaded")

if not os.path.exists("/kaggle/working/SKANN-SSL"):
    !git clone --depth 1 https://github.com/suniltyagi/SKANN-SSL.git
    log("Repository cloned (shallow)")
else:
    log("Repository already exists")

tensor_dir = "/kaggle/working/SKANN-SSL/data/prototype_dataset/tensors/"
num_files = len(os.listdir(tensor_dir)) if os.path.exists(tensor_dir) else 0
log(f"Found {num_files} tensor files")

gc.collect()
log("🎯 Data ingestion complete")

### Training script (generated in-notebook)

- Writes the training script used by the notebook. **Note:** in the repo mainline, the canonical training entrypoint is `stages/stage3_ssl/train_script.py`. This notebook-generated script should be kept in sync or replaced by importing the canonical script when you are ready.


In [ ]:
%%writefile train_script_v2.py
"""
SKANN-SSL V2.1.0 Training Script
================================
UNDERWATER-APPROPRIATE KERNEL SIZES

Key Changes from V2.0.x:
1. SK kernels: (3,5,7,11,15) → (31, 63, 127, 255, 511, 1023)
   - Now captures shaft_rate, blade_pass, generator, resonance, cavitation
2. Projector: 512→1024→128 → 512→4096→8192→128 (restored V1 size)
3. Maintains: SyncBatchNorm for DDP compatibility

Frequency Coverage @ 16kHz:
  k=31   → 500+ Hz (cavitation)
  k=63   → 250+ Hz (resonance)
  k=127  → 125+ Hz (blade pass)
  k=255  → 62+ Hz  (generator 50Hz)
  k=511  → 31+ Hz  (generator 25Hz)
  k=1023 → 15+ Hz  (shaft rate)
"""

import os
import sys
import time
import datetime
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP


# ============================================================================
# LOGGING UTILITIES
# ============================================================================

def log(msg, rank=None):
    ts = time.strftime('%H:%M:%S')
    if rank is not None:
        print(f"[{ts}][R{rank}] {msg}", flush=True)
    else:
        print(f"[{ts}] {msg}", flush=True)


def log_crash(work_dir, exc, rank=0):
    os.makedirs(work_dir, exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(work_dir, f"crash_rank{rank}_{ts}.txt")
    msg = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"Rank: {rank}\n")
        f.write(f"Timestamp: {datetime.datetime.now().isoformat()}\n")
        f.write("=" * 50 + "\n")
        f.write(msg)
    print(f"\n🚨 CRASH LOG WRITTEN: {path}")
    print(msg)
    return path


def write_heartbeat(work_dir, epoch, step, loss, rank=0):
    os.makedirs(work_dir, exist_ok=True)
    path = os.path.join(work_dir, f"heartbeat_rank{rank}.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"epoch={epoch}, step={step}, loss={loss:.6f}\n")
        f.write(f"timestamp={datetime.datetime.now().isoformat()}\n")


class RankLogger:
    def __init__(self, work_dir, rank):
        os.makedirs(work_dir, exist_ok=True)
        self.path = os.path.join(work_dir, f"rank{rank}.log")
        self.rank = rank
        with open(self.path, "w") as f:
            f.write(f"=== Rank {rank} Log Started: {datetime.datetime.now().isoformat()} ===\n")
    
    def log(self, msg):
        ts = time.strftime('%H:%M:%S')
        line = f"[{ts}] {msg}\n"
        print(f"[R{self.rank}] {msg}", flush=True)
        with open(self.path, "a") as f:
            f.write(line)


# ============================================================================
# MODEL COMPONENTS - V2.1.0 WITH UNDERWATER-APPROPRIATE KERNELS
# ============================================================================

def _norm_1d(channels, kind='gn', groups=8):
    """1D normalization layer factory."""
    if kind == 'bn':
        return nn.BatchNorm1d(channels)
    if kind == 'ln':
        return nn.GroupNorm(1, channels)
    return nn.GroupNorm(min(groups, channels), channels)


class SKConv1D(nn.Module):
    """
    Selective Kernel 1D Convolution - V2.1.0
    
    UNDERWATER-APPROPRIATE KERNEL SIZES:
    Default kernels (31, 63, 127, 255, 511, 1023) capture:
    - Cavitation: 400-6300 Hz (k=31)
    - Resonance: 50-500 Hz (k=31-127)
    - Blade pass: 3-150 Hz (k=127-1023)
    - Generator: 25, 50 Hz (k=255-1023)
    - Shaft rate: 15-30 Hz (k=1023)
    """
    
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        kernel_sizes: tuple = (31, 63, 127, 255, 511, 1023),  # V2.1.0: Underwater-appropriate!
        stride: int = 1,
        reduction: int = 16,
        norm: str = 'gn',
        act: str = 'gelu',
        residual: bool = True,
        dropout: float = 0.0
    ):
        super().__init__()
        
        # Multi-branch convolutions with LARGE kernels for low frequencies
        self.branches = nn.ModuleList()
        for k in kernel_sizes:
            pad = k // 2
            self.branches.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=k, stride=stride, 
                         padding=pad, bias=False)
            )
        
        self.n_branches = len(kernel_sizes)
        self.out_ch = out_ch
        self.kernel_sizes = kernel_sizes
        
        # Attention MLP
        hidden = max(out_ch // reduction, 8)
        self.fc1 = nn.Linear(out_ch, hidden)
        self.fc2 = nn.Linear(hidden, out_ch * self.n_branches)
        
        # Normalization and activation (GroupNorm for DDP safety)
        self.norm = _norm_1d(out_ch, norm)
        self.act = nn.GELU() if act == 'gelu' else nn.ReLU(inplace=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        # Residual connection
        self.residual = residual
        self.match = None
        if residual and (in_ch != out_ch or stride != 1):
            self.match = nn.Conv1d(in_ch, out_ch, kernel_size=1, 
                                   stride=stride, bias=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute all branch outputs (each with different temporal scale)
        feats = [branch(x) for branch in self.branches]
        
        # Sum for global descriptor
        U = torch.stack(feats, dim=1).sum(dim=1)
        
        # Channel attention: global avg pool → FC → softmax
        s = F.adaptive_avg_pool1d(U, 1).squeeze(-1)
        z = self.fc2(F.relu(self.fc1(s), inplace=False))
        a = z.view(z.size(0), self.n_branches, self.out_ch)
        a = F.softmax(a, dim=1).unsqueeze(-1)
        
        # Weighted fusion of multi-scale features
        feats_stacked = torch.stack(feats, dim=1)
        V = (a * feats_stacked).sum(dim=1)
        
        # Norm + activation + dropout
        out = self.norm(V)
        out = self.act(out)
        out = self.dropout(out)
        
        # Residual
        if self.residual:
            res = x if self.match is None else self.match(x)
            out = out + res
        
        return out


class SKFilterbank(nn.Module):
    """
    Selective Kernel Filterbank - V2.1.0
    
    Converts raw waveform to time-feature map using
    underwater-appropriate multi-scale kernels.
    """
    
    def __init__(
        self,
        out_ch: int = 64,
        kernel_sizes: tuple = (31, 63, 127, 255, 511, 1023),
        norm: str = 'gn'
    ):
        super().__init__()
        
        self.stem = SKConv1D(
            in_ch=1,
            out_ch=out_ch,
            kernel_sizes=kernel_sizes,
            stride=1,
            reduction=16,
            norm=norm,
            act='gelu',
            residual=False  # No residual for 1->64 expansion
        )
        self.post_norm = _norm_1d(out_ch, norm)
        
        # Log kernel info
        print(f"    SKFilterbank kernels: {kernel_sizes}")
        print(f"    Frequency coverage @ 16kHz: {16000/kernel_sizes[-1]:.0f}Hz - {16000/kernel_sizes[0]:.0f}Hz")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        return self.post_norm(h)


class HybridSKEncoderV2(nn.Module):
    """
    V2.1.0 Encoder - Underwater-Appropriate Architecture
    
    Key changes from V2.0.x:
    1. SK kernels: (31, 63, 127, 255, 511, 1023) for low-freq capture
    2. Large projector: 512→4096→8192→128 (matches V1)
    3. SyncBatchNorm compatible (standard BatchNorm2d)
    """
    
    def __init__(self, latent_dim=128):
        super().__init__()
        
        print("  Building HybridSKEncoderV2 (V2.1.0)...")
        
        # SK Frontend with UNDERWATER-APPROPRIATE kernels
        self.sk_frontend = SKFilterbank(
            out_ch=64,
            kernel_sizes=(31, 63, 127, 255, 511, 1023)  # V2.1.0!
        )
        self.downsample = nn.AvgPool1d(kernel_size=8, stride=8)
        
        # Channel bridge (GroupNorm for safety)
        self.channel_bridge = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=False)
        )
        
        # 2D Backbone with standard BatchNorm2d (will become SyncBN)
        self.backbone2d = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=False),
            nn.Conv2d(64, 128, 3, padding=1, stride=(2, 2)),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=False),
            nn.Conv2d(128, 256, 3, padding=1, stride=(2, 1)),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=False),
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=False),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(1)
        )
        
        # LARGE PROJECTOR - restored to V1 size!
        # This is critical for Barlow Twins
        self.projector = nn.Sequential(
            nn.Linear(512, 4096),
            nn.LayerNorm(4096),
            nn.ReLU(inplace=False),
            nn.Linear(4096, 8192),
            nn.LayerNorm(8192),
            nn.ReLU(inplace=False),
            nn.Linear(8192, latent_dim)
        )
        
        print(f"    Projector: 512 → 4096 → 8192 → {latent_dim}")
        self._count_params()
    
    def _count_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Total params: {total/1e6:.1f}M, Trainable: {trainable/1e6:.1f}M")
    
    def forward(self, x):
        # Robust shape handling
        if x.dim() > 3:
            x = x.view(x.size(0), -1).unsqueeze(1)
        elif x.dim() == 2:
            x = x.unsqueeze(1)
        
        # SK Frontend (multi-scale temporal features)
        x = self.sk_frontend(x)
        x = self.downsample(x)
        
        # Bridge to 2D
        x = self.channel_bridge(x)
        x = x.unsqueeze(1)
        
        # 2D Backbone
        x = self.backbone2d(x)
        
        # Project to latent
        return self.projector(x)


# ============================================================================
# DATASET
# ============================================================================

class HierarchicalDataset(Dataset):
    def __init__(self, manifest_path, data_dir=None):
        self.df = pd.read_csv(manifest_path)
        self.data_dir = data_dir or '/kaggle/working/SKANN-SSL/data/prototype_dataset/tensors/'
        self.class_to_id = {c: i for i, c in enumerate(sorted(self.df["vessel_class"].unique()))}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        anchor_id = int(row['anchor_clip_id'])
        p_ids = str(row['partner_clip_ids']).split('|')
        p_ids = [int(x) for x in p_ids if x.strip()]
        partner_id = np.random.choice(p_ids)
        
        y1 = np.load(os.path.join(self.data_dir, f"tensor_{anchor_id:06d}.npy")).flatten()
        y2 = np.load(os.path.join(self.data_dir, f"tensor_{partner_id:06d}.npy")).flatten()
        
        return (
            torch.from_numpy(y1).float(),
            torch.from_numpy(y2).float(),
            self.class_to_id[row["vessel_class"]]
        )


# ============================================================================
# BARLOW TWINS LOSS
# ============================================================================

def barlow_twins_loss(z1, z2, lambd=5e-3):
    """Barlow Twins loss - no inplace operations."""
    batch_size = z1.size(0)
    
    # Normalize
    z1_mean = z1.mean(dim=0)
    z1_std = z1.std(dim=0) + 1e-6
    z1_norm = (z1 - z1_mean) / z1_std
    
    z2_mean = z2.mean(dim=0)
    z2_std = z2.std(dim=0) + 1e-6
    z2_norm = (z2 - z2_mean) / z2_std
    
    # Cross-correlation matrix
    c = torch.mm(z1_norm.T, z2_norm) / batch_size
    
    # Loss computation (no inplace)
    diag = torch.diagonal(c)
    on_diag_loss = torch.pow(1.0 - diag, 2).sum()
    c_squared = torch.pow(c, 2)
    off_diag_loss = c_squared.sum() - torch.pow(diag, 2).sum()
    
    return on_diag_loss + lambd * off_diag_loss


# ============================================================================
# DDP TRAINING WORKER (with SyncBatchNorm)
# ============================================================================

def train_worker(rank, world_size, manifest_path, data_dir, epochs=50, batch_size=4):
    log_dir = "/kaggle/working/run_logs"
    logger = RankLogger(log_dir, rank)
    
    try:
        logger.log("Initializing process group")
        torch.cuda.set_device(rank)
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        dist.init_process_group("nccl", rank=rank, world_size=world_size)
        logger.log("Process group initialized")
        
        logger.log("Loading dataset")
        dataset = HierarchicalDataset(manifest_path, data_dir)
        sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
        loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                           num_workers=2, pin_memory=True, drop_last=True)
        logger.log(f"Dataset loaded: {len(dataset)} samples")
        
        # Build model with SyncBatchNorm
        logger.log("Building model (V2.1.0 - Underwater kernels + Large projector)")
        model = HybridSKEncoderV2(latent_dim=128)
        
        # Convert BatchNorm → SyncBatchNorm for DDP
        model = nn.SyncBatchNorm.convert_sync_batchnorm(model)
        logger.log("Converted BatchNorm → SyncBatchNorm")
        
        model = model.cuda(rank)
        model = DDP(model, device_ids=[rank])
        logger.log("Model wrapped in DDP")
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        logger.log("Optimizer ready")
        
        if rank == 0:
            with open('/kaggle/working/loss_history.txt', 'w') as f:
                f.write("epoch,loss\n")
        
        logger.log(f"Starting training: {epochs} epochs")
        
        for epoch in range(1, epochs + 1):
            sampler.set_epoch(epoch)
            model.train()
            total_loss = 0.0
            
            logger.log(f"Epoch {epoch}/{epochs} started")
            
            for step, (y1, y2, _) in enumerate(loader):
                y1 = y1.cuda(rank, non_blocking=True)
                y2 = y2.cuda(rank, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                
                # Dual forward passes
                emb1 = model(y1)
                emb2 = model(y2)
                loss = barlow_twins_loss(emb1, emb2, lambd=5e-3)
                
                loss.backward()
                optimizer.step()
                
                step_loss = loss.item()
                total_loss += step_loss
                
                if step % 10 == 0:
                    write_heartbeat(log_dir, epoch, step, step_loss, rank)
            
            scheduler.step()
            avg_loss = total_loss / len(loader)
            logger.log(f"Epoch {epoch}/{epochs} complete | Loss: {avg_loss:.4f}")
            
            if rank == 0:
                with open('/kaggle/working/loss_history.txt', 'a') as f:
                    f.write(f"{epoch},{avg_loss:.4f}\n")
                    f.flush()
                    os.fsync(f.fileno())
                
                if epoch % 10 == 0 or epoch == epochs:
                    save_path = f"/kaggle/working/BT_ckpt_epoch_{epoch:03d}.pth"
                    torch.save(
                        {"epoch": epoch, "encoder": model.module.state_dict()},
                        save_path
                    )
                    logger.log(f"Checkpoint saved: {save_path}")
        
        if rank == 0:
            torch.save(model.module.state_dict(), "/kaggle/working/SKANN_SSL_V2_Final.pth")
            logger.log("✅ Training complete! Final model saved.")
        
        dist.destroy_process_group()
        logger.log("Process group destroyed")
        
    except Exception as e:
        log_crash(log_dir, e, rank)
        if dist.is_initialized():
            dist.destroy_process_group()
        raise


# ============================================================================
# SINGLE GPU TRAINING (Fallback)
# ============================================================================

def train_single_gpu(manifest_path, data_dir, epochs=50, batch_size=4):
    log_dir = "/kaggle/working/run_logs"
    os.makedirs(log_dir, exist_ok=True)
    
    try:
        log("Single GPU Training Mode (V2.1.0)")
        
        log("Loading dataset")
        dataset = HierarchicalDataset(manifest_path, data_dir)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                           num_workers=2, drop_last=True)
        log(f"Dataset loaded: {len(dataset)} samples")
        
        log("Building model")
        model = HybridSKEncoderV2(latent_dim=128).cuda()
        log("Model built")
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        log("Optimizer ready")
        
        with open('/kaggle/working/loss_history.txt', 'w') as f:
            f.write("epoch,loss\n")
        
        log(f"Starting training: {epochs} epochs")
        
        for epoch in range(1, epochs + 1):
            model.train()
            total_loss = 0.0
            
            log(f"Epoch {epoch}/{epochs} started")
            
            for step, (y1, y2, _) in enumerate(loader):
                y1, y2 = y1.cuda(), y2.cuda()
                optimizer.zero_grad(set_to_none=True)
                
                z1 = model(y1)
                z2 = model(y2)
                loss = barlow_twins_loss(z1, z2, lambd=5e-3)
                
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                
                if step % 10 == 0:
                    write_heartbeat(log_dir, epoch, step, loss.item(), 0)
            
            scheduler.step()
            avg_loss = total_loss / len(loader)
            log(f"Epoch {epoch}/{epochs} complete | Loss: {avg_loss:.4f}")
            
            with open('/kaggle/working/loss_history.txt', 'a') as f:
                f.write(f"{epoch},{avg_loss:.4f}\n")
            
            if epoch % 10 == 0 or epoch == epochs:
                torch.save({"epoch": epoch, "encoder": model.state_dict()},
                          f"/kaggle/working/BT_ckpt_epoch_{epoch:03d}.pth")
                log(f"Checkpoint saved: epoch {epoch}")
        
        torch.save(model.state_dict(), "/kaggle/working/SKANN_SSL_V2_Final.pth")
        log("✅ Training complete!")
        
    except Exception as e:
        log_crash(log_dir, e, 0)
        raise

### Launch Training

- Runs the SSL training (Barlow Twins) using the generated training script, saves checkpoints and the final `.pth` weights to `/kaggle/working/`.


In [ ]:
# Cell 4: Launch Training
import os
import random
import torch
import torch.multiprocessing as mp
from train_script_v2 import train_worker, train_single_gpu, log_crash

def launch():
    log("🚀 Launching SKANN-SSL V2.1.0 Training...")
    print("=" * 70)
    print("  V2.1.0 - UNDERWATER-APPROPRIATE KERNELS")
    print("  ")
    print("  SK Kernels: (31, 63, 127, 255, 511, 1023)")
    print("  - k=31:   captures cavitation (500+ Hz)")
    print("  - k=63:   captures resonance (250+ Hz)")
    print("  - k=127:  captures blade pass (125+ Hz)")
    print("  - k=255:  captures generator 50Hz (62+ Hz)")
    print("  - k=511:  captures generator 25Hz (31+ Hz)")
    print("  - k=1023: captures shaft rate (15+ Hz)")
    print("  ")
    print("  Projector: 512 → 4096 → 8192 → 128 (restored V1 size)")
    print("=" * 70)
    
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(random.randint(29500, 29999))
    
    torch.cuda.empty_cache()
    world_size = torch.cuda.device_count()
    log(f"GPUs detected: {world_size}")
    
    manifest = "/kaggle/working/pairing_manifest.csv"
    data_dir = "/kaggle/working/SKANN-SSL/data/prototype_dataset/tensors/"
    log_dir = "/kaggle/working/run_logs"
    
    try:
        if world_size < 2:
            log("⚠️ Single GPU mode (no DDP/SyncBN needed)")
            train_single_gpu(manifest, data_dir, epochs=50, batch_size=4)
        else:
            log(f"🚀 DDP mode with {world_size} GPUs + SyncBatchNorm")
            mp.spawn(
                train_worker,
                args=(world_size, manifest, data_dir, 50, 4),
                nprocs=world_size,
                join=True
            )
        log("✅ Training complete!")
        
    except Exception as e:
        log_crash(log_dir, e, rank=-1)
        raise

launch()

### Extract Embeddings

- Loads the trained encoder and computes embeddings for the dataset to enable downstream cluster-quality diagnostics.


In [ ]:
# Cell 6: Extract Embeddings
import torch
import numpy as np
import os
import glob
import pandas as pd
from train_script_v2 import HybridSKEncoderV2

def extract_embeddings():
    log("🔬 Extracting embeddings...")
    
    weights = "/kaggle/working/SKANN_SSL_V2_Final.pth"
    if not os.path.exists(weights):
        cands = sorted(glob.glob("/kaggle/working/BT_ckpt_epoch_*.pth"))
        if not cands:
            log("❌ No weights found!")
            return None, None
        weights = cands[-1]
        log(f"Using checkpoint: {weights}")
    
    model = HybridSKEncoderV2(latent_dim=128)
    state = torch.load(weights, map_location='cpu')
    if 'encoder' in state:
        state = state['encoder']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state)
    model.eval().cuda()
    log("Model loaded")
    
    tensor_dir = '/kaggle/working/SKANN-SSL/data/prototype_dataset/tensors/'
    files = sorted(glob.glob(os.path.join(tensor_dir, 'tensor_*.npy')))
    
    manifest = pd.read_csv('/kaggle/working/pairing_manifest.csv')
    clip_to_class = dict(zip(manifest['anchor_clip_id'], manifest['vessel_class']))
    class_to_id = {c: i for i, c in enumerate(sorted(manifest['vessel_class'].unique()))}
    
    embeddings, labels = [], []
    with torch.no_grad():
        for f in files:
            clip_id = int(os.path.basename(f).replace('tensor_', '').replace('.npy', ''))
            x = torch.from_numpy(np.load(f).flatten()).float().unsqueeze(0).cuda()
            z = model(x).cpu().numpy().flatten()
            embeddings.append(z)
            labels.append(class_to_id.get(clip_to_class.get(clip_id, ''), -1))
    
    log(f"✅ Extracted {len(embeddings)} embeddings")
    return np.array(embeddings), np.array(labels)

embeddings, labels = extract_embeddings()

### Silhouette Analysis

- Computes the Silhouette Score for the embedding space as an unsupervised quality metric (used to compare V1 vs V2.1.0).


In [ ]:
# Cell 7: Silhouette Analysis
from sklearn.metrics import silhouette_score, silhouette_samples
import numpy as np

silhouette = None
sample_scores = None

if embeddings is not None:
    log("📊 Computing Silhouette Score...")
    mask = labels >= 0
    
    if mask.sum() > 10:
        silhouette = silhouette_score(embeddings[mask], labels[mask], metric='cosine')
        sample_scores = silhouette_samples(embeddings[mask], labels[mask], metric='cosine')
        
        print("\n" + "=" * 70)
        print(f"📊 V2.1.0 Silhouette Score (cosine): {silhouette:.4f}")
        print(f"   V1 Baseline:                      0.3997")
        print(f"   V2.0.8 (wrong kernels):          -0.1253")
        print(f"   Target:                           >0.42")
        print("=" * 70)
        
        if silhouette > 0.3997:
            print("\n✅ V2.1.0 EXCEEDS V1 BASELINE! SK mechanism working!")
        elif silhouette > 0:
            print(f"\n⚠️ V2.1.0 positive ({silhouette:.4f}) but below V1 baseline")
        else:
            print("\n❌ V2.1.0 still negative - may need V2.2.0 (hierarchical RF)")
        
        log(f"Per-sample: min={sample_scores.min():.4f}, max={sample_scores.max():.4f}")

### Silhouette Distribution Plot

- Plots per-sample silhouette values to inspect cluster separability and identify problematic regions/classes.


In [ ]:
# Cell 8: Silhouette Distribution Plot
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if sample_scores is not None:
    log("📊 Plotting silhouette distribution...")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(sample_scores, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(x=silhouette, color='red', linestyle='--', linewidth=2, label=f'V2.1.0: {silhouette:.4f}')
    axes[0].axvline(x=0.3997, color='green', linestyle=':', linewidth=2, label='V1: 0.3997')
    axes[0].axvline(x=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
    axes[0].set_xlabel('Silhouette Score')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Per-Sample Silhouette Distribution (V2.1.0)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Per-class boxplot
    manifest = pd.read_csv('/kaggle/working/pairing_manifest.csv')
    class_names = sorted(manifest['vessel_class'].unique())
    mask = labels >= 0
    unique_labels = np.unique(labels[mask])
    class_scores = [sample_scores[labels[mask] == l] for l in unique_labels]
    
    bp = axes[1].boxplot(class_scores, labels=class_names, patch_artist=True)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(class_names)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1].axhline(y=silhouette, color='red', linestyle='--', label=f'V2.1.0: {silhouette:.4f}')
    axes[1].axhline(y=0.3997, color='green', linestyle=':', label='V1: 0.3997')
    axes[1].axhline(y=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
    axes[1].set_xlabel('Vessel Class')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Per-Class Silhouette (V2.1.0 Underwater Kernels)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig("/kaggle/working/silhouette_distribution.png", dpi=300)
    plt.show()
    log("✅ Silhouette plot saved")

### UMAP Visualization

- Makes visualisation cells resilient to kernel restarts by reloading `embeddings.npy`, `labels.npy`, and `silhouette.txt` from disk if they are not present in memory. **No restart happens here.**


In [ ]:
# --- Sanitisation: restart-safe load of embeddings/labels/silhouette (no training changes) ---
import os
import numpy as np
if 'embeddings' not in globals() and os.path.exists('/kaggle/working/embeddings.npy'):
    embeddings = np.load('/kaggle/working/embeddings.npy')
if 'labels' not in globals() and os.path.exists('/kaggle/working/labels.npy'):
    labels = np.load('/kaggle/working/labels.npy')
if 'silhouette' not in globals() and os.path.exists('/kaggle/working/silhouette.txt'):
    try:
        silhouette = float(open('/kaggle/working/silhouette.txt').read().strip())
    except Exception:
        pass

# Cell 9: UMAP Visualization
import umap
import matplotlib.pyplot as plt
import pandas as pd

umap_config = None
emb_2d = None

if embeddings is not None:
    log("🎨 Generating UMAP...")
    
    umap_config = {'n_neighbors': 15, 'min_dist': 0.1, 'metric': 'cosine', 'random_state': 42}
    reducer = umap.UMAP(**umap_config)
    emb_2d = reducer.fit_transform(embeddings)
    
    manifest = pd.read_csv('/kaggle/working/pairing_manifest.csv')
    class_names = sorted(manifest['vessel_class'].unique())
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=labels, cmap='Spectral',
                         s=40, alpha=0.7, edgecolors='white', linewidth=0.5)
    cbar = plt.colorbar(scatter, ticks=range(len(class_names)))
    cbar.ax.set_yticklabels(class_names)
    cbar.set_label('Vessel Class')
    
    title = f"SKANN-SSL V2.1.0: Underwater-Appropriate Kernels\n"
    title += f"SK kernels: (31, 63, 127, 255, 511, 1023) | Silhouette: {silhouette:.4f}"
    plt.title(title)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.savefig("/kaggle/working/vessel_clusters_umap.png", dpi=300, bbox_inches='tight')
    plt.show()
    log("✅ UMAP saved")

### Export Production Bundle

- Creates the `SKANN_SSL_Production_Bundle.joblib` containing model weights, class labels, and key metadata for downstream use. This artefact is kept locally and is not committed to git.


In [ ]:
# Cell 10: Export Production Bundle
from datetime import datetime
import joblib
import torch
import pandas as pd
import glob
import os

def export_bundle():
    log("📦 Exporting production bundle...")
    
    weights = "/kaggle/working/SKANN_SSL_V2_Final.pth"
    if not os.path.exists(weights):
        cands = sorted(glob.glob("/kaggle/working/BT_ckpt_epoch_*.pth"))
        if not cands:
            log("❌ No weights!")
            return
        weights = cands[-1]
    
    ckpt = torch.load(weights, map_location='cpu')
    state = ckpt['encoder'] if 'encoder' in ckpt else ckpt
    state = {k.replace('module.', ''): v for k, v in state.items()}
    
    manifest = pd.read_csv('/kaggle/working/pairing_manifest.csv')
    vessel_labels = sorted(manifest['vessel_class'].unique())
    
    bundle = {
        'model_state': state,
        'vessel_labels': vessel_labels,
        'class_map': {l: i for i, l in enumerate(vessel_labels)},
        'umap_config': umap_config,
        'metrics': {
            'silhouette_score': float(silhouette) if silhouette else None,
            'v1_baseline': 0.3997,
            'n_samples': len(embeddings) if embeddings is not None else None,
        },
        'metadata': {
            'version': 'v2.1.0',
            'architecture': 'HybridSKEncoderV2',
            'sk_kernels': (31, 63, 127, 255, 511, 1023),
            'projector': '512→4096→8192→128',
            'latent_dim': 128,
            'key_fix': 'Underwater-appropriate kernel sizes',
            'export_date': datetime.now().isoformat(),
        }
    }
    
    path = '/kaggle/working/SKANN_SSL_V2_Production_Bundle.joblib'
    joblib.dump(bundle, path)
    
    print("\n" + "=" * 70)
    print("📦 Production Bundle V2.1.0:")
    print(f"   version: {bundle['metadata']['version']}")
    print(f"   sk_kernels: {bundle['metadata']['sk_kernels']}")
    print(f"   projector: {bundle['metadata']['projector']}")
    print(f"   silhouette: {bundle['metrics']['silhouette_score']:.4f}")
    print(f"   v1_baseline: {bundle['metrics']['v1_baseline']}")
    print("=" * 70)
    log(f"✅ Bundle saved: {os.path.getsize(path)/1e6:.1f} MB")

export_bundle()

### Inference Test

- Runs a small sanity-check inference on a random sample to verify the exported weights load correctly and produce the expected embedding shape.


In [ ]:
# Cell 11: Inference Test
import joblib
import torch
import numpy as np
import os
import random
from train_script_v2 import HybridSKEncoderV2

def inference_test():
    log("🚢 Running inference test...")
    
    bundle_path = "/kaggle/working/SKANN_SSL_V2_Production_Bundle.joblib"
    if not os.path.exists(bundle_path):
        log("❌ Bundle not found!")
        return
    
    bundle = joblib.load(bundle_path)
    model = HybridSKEncoderV2(latent_dim=128).cuda()
    model.load_state_dict(bundle["model_state"])
    model.eval()
    
    tensor_dir = "/kaggle/working/SKANN-SSL/data/prototype_dataset/tensors/"
    test_file = random.choice([f for f in os.listdir(tensor_dir) if f.endswith('.npy')])
    
    x = np.load(os.path.join(tensor_dir, test_file)).flatten()
    x = torch.from_numpy(x).float().unsqueeze(0).cuda()
    
    with torch.no_grad():
        emb = model(x)
    
    log(f"✅ Input: {test_file}")
    log(f"✅ Embedding shape: {emb.shape}")
    log(f"✅ SK kernels: {bundle['metadata']['sk_kernels']}")
    log(f"✅ Silhouette: {bundle['metrics']['silhouette_score']:.4f}")

inference_test()

### Code cell

- Persists `embeddings.npy`, `labels.npy`, and `silhouette.txt` so that plots can be regenerated even if the session restarts. This is non-fatal and lightweight (recommended).


In [ ]:
# Persist key artefacts for restart-safe diagnostics (non-fatal)
try:
    import numpy as np
    if 'embeddings' in globals():
        np.save('/kaggle/working/embeddings.npy', np.asarray(embeddings, dtype=np.float32))
        print('✅ Saved /kaggle/working/embeddings.npy')
    if 'labels' in globals():
        np.save('/kaggle/working/labels.npy', np.asarray(labels, dtype=np.int64))
        print('✅ Saved /kaggle/working/labels.npy')
    if 'silhouette' in globals():
        with open('/kaggle/working/silhouette.txt', 'w') as f:
            f.write(str(float(silhouette)))
        print('✅ Saved /kaggle/working/silhouette.txt')
except Exception as e:
    print('⚠️ Persistence skipped:', e)


### Archive & Download Link

- Bundles key outputs (weights, joblib bundle, plots, logs) into a timestamped ZIP and creates a clickable download link. Keep this near the end so the archive is complete.


In [ ]:
# Cell 13: Archive & Download Link
import os
import zipfile
import glob
from datetime import datetime
from IPython.display import display, FileLink, HTML

def create_archive():
    """Bundle all outputs into a single downloadable zip file."""
    
    log("📦 Creating archive...")
    
    # Generate timestamped filename
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_name = f"SKANN_SSL_V2.1.0_{ts}.zip"
    zip_path = f"/kaggle/working/{zip_name}"
    
    # Files to include
    patterns = [
        "/kaggle/working/*.pth",
        "/kaggle/working/*.joblib", 
        "/kaggle/working/*.png",
        "/kaggle/working/*.txt",
        "/kaggle/working/run_logs/*.log",
        "/kaggle/working/run_logs/*.txt",
    ]
    
    files_added = []
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for pattern in patterns:
            for filepath in glob.glob(pattern):
                arcname = os.path.basename(filepath)
                # Put logs in a subfolder
                if '/run_logs/' in filepath:
                    arcname = f"run_logs/{arcname}"
                zf.write(filepath, arcname)
                files_added.append(arcname)
    
    # Get zip size
    zip_size = os.path.getsize(zip_path) / 1e6
    
    print("\n" + "=" * 70)
    print(f"📦 ARCHIVE CREATED: {zip_name}")
    print(f"   Size: {zip_size:.1f} MB")
    print(f"   Files: {len(files_added)}")
    print("=" * 70)
    
    # List contents
    print("\n📁 Archive contents:")
    for f in sorted(files_added):
        print(f"   • {f}")
    
    # Create clickable download link
    print("\n" + "=" * 70)
    print("⬇️  DOWNLOAD LINK:")
    print("=" * 70)
    display(FileLink(zip_path, result_html_prefix="Click to download: "))
    
    # Also show direct path for manual access
    print(f"\n📍 Or find in sidebar: Output → {zip_name}")
    
    return zip_path

archive_path = create_archive()

### Final Summary

- Prints a final run summary (scores, key settings) and performs cleanup.


In [ ]:
# Cell 14: Final Summary
safe_cleanup()

print("\n" + "=" * 70)
print("🎯 SKANN-SSL V2.1.0 Pipeline Complete!")
print("=" * 70)

print("\n📁 Output Files:")
import glob
for cat, pattern in [
    ("Weights", "/kaggle/working/*.pth"),
    ("Bundle", "/kaggle/working/*.joblib"),
    ("Plots", "/kaggle/working/*.png"),
    ("Logs", "/kaggle/working/*.txt"),
]:
    files = glob.glob(pattern)
    if files:
        print(f"\n{cat}:")
        for f in files:
            print(f"   {os.path.basename(f)}: {os.path.getsize(f)/1e6:.2f} MB")

print("\n" + "=" * 70)
print("✅ V2.1.0 Key Changes:")
print("   SK Kernels: (31, 63, 127, 255, 511, 1023)")
print("   - Covers: cavitation, resonance, blade pass, generator, shaft rate")
print("   - Frequency range: 15 Hz - 500 Hz (underwater-appropriate)")
print("   ")
print("   Projector: 512 → 4096 → 8192 → 128 (restored V1 size)")
print("   ")
print("   SyncBatchNorm: Enabled for DDP compatibility")
print("\n📊 Results Comparison:")
print(f"   V1 Baseline:    0.3997")
print(f"   V2.0.8 (small kernels): -0.1253")
if silhouette is not None:
    print(f"   V2.1.0 (underwater):    {silhouette:.4f}")
print("=" * 70)

if silhouette is not None and silhouette < 0:
    print("\n⚠️ If V2.1.0 still underperforms, try V2.2.0:")
    print("   - Moderate SK kernels + stride")
    print("   - Rely on pooling/depth for effective RF")
    print("   - Match V1's hierarchical approach")